# colab-sglang · SGLang 一键部署（Notebook）

基于 **SGLang** 在 **Colab GPU 会话**中部署大模型，并对外提供 **OpenAI 兼容 API**。
本 Notebook 覆盖 **Colab 侧**完整流程（等价于在 Colab terminal 中依次执行 `colab.sh setup all` / `colab.sh install sglang` / `colab.sh sglang start` / `colab.sh bore start`）：

> **0 项目路径与环境检查 → 1 加载 .env → 2 安装依赖 → 3 安装并启动 → 4 等待就绪 → 5 状态检查 → 6 公网隧道 → 7 API 验证 → 8 服务管理**

**前置条件**

1. **宿主机**上已执行 `SESSION_NAME=gcloud ./colab.sh vps all`（或 `vps create` + `vps mount`）创建 GPU 会话并挂载 Drive（该步骤不在本 Notebook 内）；
2. 本项目全部文件（含 `.env`；`.env` 未入 git，请自行上传或用 `relaydrop` 传送）已传到 Colab，**位置不限**：无需放在 Notebook 工作目录（Colab 默认 `/content`），在第 0 节可指定**自定义路径**（如 Google Drive 挂载目录），留空则自动探测常见位置；
3. GPU 会话显存足够（项目目标机为 96GB，默认模型为数十 GB 量级）。

**用法**：从头 **Run All** 即可；首次运行会下载模型权重（默认模型约 56GB，"等待就绪"单元格会滚动显示进度）。
若已在 Colab terminal 中执行过对应脚本，可直接跳到 **4. 等待服务就绪**。

## 0. 项目路径与环境检查

单元格顶部可指定**项目根目录**（含 `colab.sh` / `.env` 的目录）；留空则按以下顺序自动探测：

`COLAB_PROJECT_DIR` 环境变量 → 当前工作目录（Colab 默认 `/content`）→ 其子目录（深 2）→ Google Drive 挂载（深 3）→ home 目录（深 2）

确定后校验文件齐全性与 GPU；找不到会报错并列出已尝试的位置。

In [ ]:
# ============ 项目路径配置(需要自定义时在这里填) ============
# 项目根目录 = 包含 colab.sh / sglang/launch.sh / .env 的目录
# 留空 = 自动探测(顺序: COLAB_PROJECT_DIR 环境变量 → 当前工作目录 → 子目录 → Drive 挂载 → home 目录)
PROJECT_DIR = ""   # 例: "/content/drive/MyDrive/colab-sglang" 或 "/root/colab-sglang"

import os, platform, re, subprocess
from pathlib import Path

MARKER = "colab.sh"
REQUIRED = ["colab.sh", "sglang/launch.sh"]

def _find_marker(base: Path, max_depth: int):
    """在 base 下深度优先查找含 MARKER 的目录(base 本身记为深度 0)"""
    if not base.is_dir():
        return None
    if (base / MARKER).exists():
        return base
    if max_depth <= 0:
        return None
    for child in sorted(p for p in base.iterdir() if p.is_dir() and not p.name.startswith(".")):
        hit = _find_marker(child, max_depth - 1)
        if hit:
            return hit
    return None

tried, resolved, how = [], None, ""
if PROJECT_DIR:  # 1) 手动指定
    p = Path(str(PROJECT_DIR)).expanduser()
    tried.append(p)
    if (p / MARKER).exists():
        resolved, how = p, "手动指定"

if not resolved:  # 2) 环境变量 / 当前目录
    cands = []
    if os.environ.get("COLAB_PROJECT_DIR"):
        cands.append((Path(os.environ["COLAB_PROJECT_DIR"]).expanduser(), "COLAB_PROJECT_DIR 环境变量"))
    cands.append((Path(os.getcwd()), "当前工作目录"))
    for cand, label in cands:
        tried.append(cand)
        if (cand / MARKER).exists():
            resolved, how = cand, label
            break

if not resolved:  # 3) 自动扫描常见位置
    scanned = set()
    for base, depth, label in (
        (Path(os.getcwd()), 2, "当前目录子目录"),
        (Path("/content/drive/MyDrive"), 3, "Google Drive (MyDrive)"),
        (Path("/content/drive/My Drive"), 3, "Google Drive (My Drive)"),
        (Path.home(), 2, "home 目录"),
    ):
        if not base.is_dir() or base in scanned:
            continue
        scanned.add(base)
        tried.append(base)
        hit = _find_marker(base, depth)
        if hit:
            resolved, how = hit, f"自动探测({label})"
            break

if resolved is None:
    raise FileNotFoundError(
        f"无法确定项目目录(标记文件: {MARKER})。已尝试:\n  "
        + "\n  ".join(str(t) for t in tried) + "\n"
        "请在单元格顶部 PROJECT_DIR 变量填入项目根目录后重跑。")

PROJECT_DIR = resolved
missing = [f for f in REQUIRED if not (PROJECT_DIR / f).exists()]
if missing:
    raise FileNotFoundError(f"{PROJECT_DIR} 缺少项目文件: {missing}")
print(f"✔ 项目目录: {PROJECT_DIR}  ({how})")
print(f"✔ 文件齐全: {', '.join(REQUIRED)}   (.env 存在: {(PROJECT_DIR / '.env').exists()})")

SGLANG_DIR = PROJECT_DIR / "sglang"
VENV_DIR = Path(os.environ.get("SGLANG_VENV_DIR") or "/tmp/sglang/venv")
print(f"✔ 引擎目录: {SGLANG_DIR}")
print(f"   venv    : {VENV_DIR}   (已初始化: {(VENV_DIR / 'bin/activate').exists()}; 换位置用 SGLANG_VENV_DIR)")

# ---- Python / GPU 检查 ----
print("Python:", platform.python_version())
r = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                   shell=True, capture_output=True, text=True)
if r.returncode == 0:
    print("✔ GPU:", r.stdout.strip().replace("\n", " | "))
    m = re.search(r"(\d+) MiB", r.stdout)
    if m and int(m.group(1)) < 40960:
        print(f"⚠ 显存仅 {int(m.group(1)) // 1024}GB, 可能不足以跑数十 GB 模型, 请调整 MODEL_REPO 为更小模型")
else:
    print("⚠ nvidia-smi 不可用:", (r.stderr or "").strip())

## 1. 加载环境变量（.env）

Colab terminal 里环境变量由 **direnv** 经 `.envrc` 加载（其内再 `dotenv` 读 `.env`）；
而 Notebook 单元格不走 direnv，这里直接解析 `.env` 写入内核 `os.environ`。
之后所有 `run(...)` 命令都会继承这些变量（`API_KEY` / `MODEL_REPO` / `BORE_PORT` …）。
密钥只以**掩码**形式展示，请勿在此单元格外打印 `.env` 内容。


In [ ]:
# 加载 .env 到内核 os.environ（支持 shell 格式: `export K=V` 与 `K=V`）
import os, re, subprocess
from pathlib import Path

if "PROJECT_DIR" not in globals():
    raise RuntimeError("PROJECT_DIR 未定义: 请先运行第 0 节(项目路径与环境检查)单元格")

def load_dotenv(path: Path = PROJECT_DIR / ".env") -> int:
    """解析 shell 格式的 .env 并写入 os.environ, 返回加载的变量数"""
    if not path.exists():
        print(f"⚠ 未找到 {path}: 请先创建(API_KEY 等将缺失)")
        return 0
    n = 0
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export "):].lstrip()
        key, _, val = line.partition("=")
        key = key.strip()
        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key):
            continue
        os.environ[key] = val.strip().strip('"').strip("'")
        n += 1
    return n

def mask(v: str) -> str:
    """掩码展示, 避免密钥泄漏到 Notebook 输出"""
    if not v:
        return "(未设置)"
    if len(v) <= 12:
        return "***"
    return f"{v[:4]}…{v[-4:]} (len={len(v)})"

def run(cmd: str, *, check: bool = True, stream: bool = False) -> int:
    """在项目根目录执行 bash 命令; 继承内核 os.environ(即 .env 加载的变量)
    stream=True 时逐行实时打印(安装等长任务用, 避免长时间无输出)"""
    print(f"$ {cmd}", flush=True)
    if stream:
        with subprocess.Popen(cmd, shell=True, cwd=PROJECT_DIR, env=os.environ,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
            for line in proc.stdout:
                print(line.rstrip(), flush=True)
            proc.wait()
            rc = proc.returncode
    else:
        r = subprocess.run(cmd, shell=True, cwd=PROJECT_DIR, env=os.environ,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        if r.stdout:
            print(r.stdout.rstrip())
        rc = r.returncode
    if rc != 0 and check:
        raise RuntimeError(f"命令退出码 {rc}: {cmd}")
    return rc

n = load_dotenv()
print(f"✔ 已从 .env 加载 {n} 个变量\n")
for k in sorted(k for k in os.environ
                if k.startswith(("SGLANG_", "BORE_", "RELAYDROP_", "OPENCODE_")) or k == "HF_TOKEN"):
    print(f"  {k} = {mask(os.environ[k])}")

# 与 launch.sh 取值规则保持一致: SGLANG_* 优先, 回退通用变量
MODEL_PATH = os.environ.get("SGLANG_MODEL_REPO") or os.environ.get("MODEL_REPO", "Qwen/Qwen3.8-27B")
SERVED_NAME = os.environ.get("SGLANG_SERVED_NAME") or \
    re.sub(r"/", "-", os.path.basename(MODEL_PATH).lower())
PORT = os.environ.get("SGLANG_PORT", "30000")
BASE_URL = f"http://localhost:{PORT}/v1"
API_KEY = os.environ.get("SGLANG_API_KEY") or os.environ.get("API_KEY", "")
print(f"\n模型路径    : {MODEL_PATH}")
print(f"API 模型名  : {SERVED_NAME}")
print(f"base_url    : {BASE_URL}")
if not API_KEY:
    print("⚠ API_KEY 为空: 若意图关闭鉴权请在 .env 中显式置空, 否则请先填入密钥")

## 2. 安装前置依赖（等价 `./colab.sh setup all`）

安装 `direnv` / `bore`（公网隧道）/ `relaydrop`（文件传输）/ `opencode`（CLI），约 1–3 分钟。
> 若已在 Colab terminal 中执行过 `./colab.sh setup all`，本单元格可跳过。

In [ ]:
run("bash colab.sh setup all")

## 3. 安装并启动（等价 `./colab.sh install sglang` + `./colab.sh sglang start`）

流程：安装 `uv` → 建 Python 3.12 venv（默认 `/tmp/sglang/venv`，**在项目外**，可用 `SGLANG_VENV_DIR` 改）→ 安装 SGLang（含 `FLASHINFER_CUDA_ARCH_LIST` 等已知坑修复）→ `setsid` 后台启动服务。

安装输出**实时打印**（数分钟），启动后本单元格立即返回；服务日志写入 `logs/sglang_server.log`。
首次运行会自动从 HuggingFace 下载模型权重（默认模型约 56GB）。

> 装过一次后可把下面单元格的安装行注释掉，只跑启动行。
> `.env` 里 `SGLANG_API_KEY` / `API_KEY` 都未设置时，`start` 会直接报错（显式置空可关闭鉴权）。

In [ ]:
# 安装 SGLang 环境(uv → venv → sglang; 不自动启动), 输出实时打印
run("bash colab.sh install sglang", stream=True)

# 后台启动服务(setsid 托管, 立即返回); 若服务已在运行, start 会自动跳过
run("bash colab.sh sglang start")
print("\n可在下一单元格查看进度, 或执行: !tail -20 logs/sglang_server.log")

## 4. 等待服务就绪

轮询 `http://localhost:30000/health` 直到 HTTP 200 或超时；每次等待打印日志最后一行，
可实时看到 安装 / 下载权重 / 加载模型 的进度。
首次运行耗时较长（主要是下载权重）；超时时间可在 `.env` 中设置 `WAIT_TIMEOUT`（秒，如 `WAIT_TIMEOUT=14400` 即 4 小时）调整。


In [ ]:
import time, urllib.request

HEALTH_URL = f"http://localhost:{PORT}/health"
WAIT_TIMEOUT = int(os.environ.get("WAIT_TIMEOUT", str(4 * 3600)))  # 默认 4 小时
INTERVAL = 30
t0 = time.time()
print(f"等待服务就绪: {HEALTH_URL} (超时 {WAIT_TIMEOUT // 3600}h, 每 {INTERVAL}s 检查一次)\n")

def last_log_line():
    p = PROJECT_DIR / "logs" / "sglang_server.log"
    if p.exists():
        lines = [l for l in p.read_text(errors="ignore").splitlines() if l.strip()]
        if lines:
            return f"logs/sglang_server.log: {lines[-1].strip()[:160]}"
    return "(暂无日志)"

while True:
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=3) as resp:
            print(f"✔ 服务就绪! HTTP {resp.status}, 耗时 {(time.time() - t0) / 60:.1f} 分钟")
            break
    except Exception:
        elapsed = int(time.time() - t0)
        if elapsed >= WAIT_TIMEOUT:
            raise TimeoutError(f"已等待 {WAIT_TIMEOUT // 3600}h 仍未就绪, 请检查日志: !tail -50 logs/sglang_server.log")
        print(f"[{elapsed // 60:02d}m{elapsed % 60:02d}s] {last_log_line()}")
        time.sleep(INTERVAL)

## 5. 状态检查

进程状态 + 健康检查 + 已服务的模型名 + 显存占用。


In [ ]:
run("bash colab.sh sglang status")

import json, urllib.request
req = urllib.request.Request(f"http://localhost:{PORT}/v1/models",
                             headers={"Authorization": f"Bearer {API_KEY}"})
with urllib.request.urlopen(req, timeout=5) as r:
    models = json.load(r)
print("已服务模型:", [m.get("id") for m in models.get("data", [])])

run("nvidia-smi --query-gpu=memory.used,memory.total --format=csv")

## 6. 暴露到公网（bore 隧道）

`bore` 把本地 `30000` 反向代理到公网端口 `BORE_PORT`（默认 `65535`，在 `.envrc` 中定义，可在 `.env` 覆盖），
同样以 setsid 后台托管。启动后从 `logs/bore.log` 解析出**公网地址**，对外 API 入口为 `<公网地址>/v1`（需带 `Authorization: Bearer <API_KEY>`）。
> 公网地址仅在 Colab 会话与隧道存活期间有效；隧道重启后地址会变化。


In [ ]:
import re, time

run("bash colab.sh bore start")
time.sleep(2)

log_path = PROJECT_DIR / "logs" / "bore.log"
bore_log = log_path.read_text(errors="ignore") if log_path.exists() else ""
urls = [u.rstrip(".,;") for u in re.findall(r"\bhttps?://\S+", bore_log)
        if "localhost" not in u and "127.0.0.1" not in u]
PUBLIC_URL = urls[-1] if urls else None

if PUBLIC_URL:
    print(f"✔ 公网地址: {PUBLIC_URL}")
    print(f"  对外 base_url: {PUBLIC_URL.rstrip('/')}/v1")
    print(f"  示例: curl {PUBLIC_URL.rstrip('/')}/v1/models -H 'Authorization: Bearer <API_KEY>'")
else:
    print("⚠ 日志中未解析到公网地址, 请手动查看 bore.log (确认 .env 中 BORE_SECRET / BORE_SERVER 正确):")
    print(bore_log[-2000:] or "(logs/bore.log 为空)")

## 7. 调用 API 验证

OpenAI 兼容协议：`base_url = http://localhost:30000/v1`，密钥为 `API_KEY`，
请求中 `model` 字段须为第 1 节打印的 **served 名**。
思考模型的思考过程在 `reasoning_content` 中返回（最终回答在 `content`）。
如需走公网验证，把下面单元格里的 `BASE_URL` 换成第 6 节得到的公网地址（含 `/v1`）即可。


In [ ]:
# OpenAI 客户端(Colab 一般预装 openai; 缺失时自动安装)
try:
    from openai import OpenAI
except ImportError:
    run("pip install -q openai")
    from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

resp = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "你好!请用一两句话介绍你自己。"}],
    temperature=0.6,
    top_p=0.95,
)
msg = resp.choices[0].message
print("【思考过程】")
print(getattr(msg, "reasoning_content", None) or "(无)")
print("\n【回答】")
print(msg.content)
print("\n【用量】", resp.usage)


In [ ]:
# 流式输出
stream = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "写一首关于 GPU 集群的短诗"}],
    temperature=0.8,
    max_tokens=512,
    stream=True,
)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()


In [ ]:
# 工具调用(解析器在启动时按模型家族自动推导, 如 qwen3_coder / deepseek_v3 / muse)
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "查询指定城市的当前天气",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "城市名, 如上海"}},
            "required": ["city"],
        },
    },
}]

resp = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "上海今天天气怎么样?"}],
    tools=tools,
    temperature=0.2,
)
msg = resp.choices[0].message
print("【回答】", msg.content)
if getattr(msg, "tool_calls", None):
    for tc in msg.tool_calls:
        print(f"【工具调用】{tc.function.name}({tc.function.arguments})")
else:
    print("(未返回 tool_calls; 该模型/解析器可能不支持 OpenAI 工具调用协议)")


In [ ]:
# 可选: 单次请求关闭思考(对支持 enable_thinking 的模型生效, 如 qwen 家族)
resp = client.chat.completions.create(
    model=SERVED_NAME,
    messages=[{"role": "user", "content": "1+1 等于几?只给答案。"}],
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content)


## 8. 服务管理速查

| 命令 | 说明 |
|---|---|
| `./colab.sh sglang start` | 后台启动服务（setsid，SSH/内核断开不受影响） |
| `./colab.sh sglang stop` / `restart` | 优雅停止 / 重启 |
| `./colab.sh sglang status` | 进程 + 健康检查 |
| `./colab.sh sglang logs` | 实时跟踪服务日志 |
| `./colab.sh sglang keep` | 守护模式：崩溃自动拉起（每 30s 检查） |
| `./colab.sh sglang bench -n 32` | 并发压测（根目录 `bench.py`，参数透传） |
| `./colab.sh bore start/stop/restart/status/logs` | 公网隧道管理 |
| `./colab.sh install sglang` | 重装环境（venv 默认在项目外 `/tmp/sglang/venv`） |

> 上面命令都从**项目根目录**执行（`colab.sh sglang ...` 会经 direnv 加载 `sglang/.envrc`，
> 等价于 `cd sglang && ./launch.sh ...`）。

运维：

```bash
tail -f ./logs/sglang_server.log   # 服务日志
curl http://localhost:30000/health # 健康检查
curl http://localhost:30000/metrics# Prometheus 指标
nvidia-smi                         # 显存占用
```

> Colab 会话被回收后容器内容丢失（Drive 挂载除外），重新进入会话后从 **3. 一键安装并启动**（或直接 Run All）重建即可。

In [ ]:
# 常用管理命令(需要时取消对应行的注释再运行; 均在项目根目录执行)
run("bash colab.sh sglang status")
# run("bash colab.sh sglang restart")   # 重启服务
# run("bash colab.sh sglang stop")      # 停止服务
# run("bash colab.sh sglang logs")      # 跟踪服务日志(长驻, 需手动中断)
# run("bash colab.sh bore status")      # 隧道状态
# run("bash colab.sh bore restart")     # 重建隧道(公网地址会变)

# 守护模式: 崩溃自动拉起(每 30s 检查), 独立后台运行, 不受本内核停止影响
# run("setsid bash colab.sh sglang keep >> logs/keeper.log 2>&1 </dev/null & sleep 1; "
#     "pgrep -f 'colab.sh sglang [k]eep' >/dev/null && echo 'keep 已启动' || echo 'keep 未启动'")
# 停止守护模式:
# run("pkill -f 'colab.sh sglang [k]eep'")